# D1-S1-Nb1: Step-by-step Build of our Workcell

Showcase: how to take your first steps in PLR

## Set your `protocol_mode`

...so it is easy to switch between simulation and execution

In [1]:
star_simulation = True  # True or False

run_identifier = "nb1"

In [2]:
star_simulation

True

## Library Import

In [3]:
import datetime
import logging
import time


---
# Create a Facility resource
...so everything that follows has one highest-level reference frame

In [4]:
from pylabrobot.resources import Coordinate, Resource
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

In [5]:
facility = Facility(name="facility", size_x=2_400, size_y=1_000, size_z=0)
facility

Facility(name='facility', location=Coordinate(000.000, 000.000, 000.000), size_x=2400, size_y=1000, size_z=0, category=facility)

In [6]:
viewer = Viewer3D(
    root = facility,
    name="nb1_first_steps"
    )

await viewer.start()

viewer on http://127.0.0.1:1340/#token=JUGqv_21yMBxChP7tmmpoLyjuqMRQq-qIQbcybMJL3U  (websocket 2124)


viewer: a browser connected, drawing with WebGPU


2026-09-28 15:04:37,667 - pylabrobot.visualizer3D.server - WARNING - refused a websocket without this viewer's token: a page left open from an earlier viewer, or one opened without the link it printed. Repeats are logged at debug.


In [7]:
from pylabrobot.resources import ContainerRack, ResourceHolder

In [8]:
from typing import List
import math 

def get_circle_positions(
  self, n: int, radius: float, *, z: float = 0, start_angle: float = 0
) -> List[Coordinate]:
  """Get n points evenly spaced on a circle around the xy-center, relative to LFB.

  Args:
    n: number of points.
    radius: circle radius in mm; must fit inside the resource footprint.
    z: height of the points above the bottom, in mm.
    start_angle: angle of the first point in degrees; 0 is +x (right), counterclockwise.
  """
  if n < 1:
    raise ValueError(f"n must be at least 1, got {n}")
  if not 0 <= radius <= min(self.get_size_x(), self.get_size_y()) / 2:
    raise ValueError(f"radius {radius} does not fit in {self.name}")
  c = self.center()
  return [
    Coordinate(
      x=c.x + radius * math.cos(math.radians(start_angle + i * 360 / n)),
      y=c.y + radius * math.sin(math.radians(start_angle + i * 360 / n)),
      z=z,
    )
    for i in range(n)
  ]

In [9]:
dummy = Resource(
    name="weird arc-shaped tubeholder",
    size_x=300,
    size_y=300,
    size_z=160.0
)



In [10]:
coordinates = get_circle_positions(dummy, n=12, radius=120.0)
coordinates = [c+Coordinate(0,0,10) for c in coordinates]

In [11]:
container_holders = [
    ResourceHolder(name=f"holder_{idx}", size_x=15, size_y=15, size_z=0).at(location)
    for idx, location in enumerate(coordinates)
    ]

In [12]:
diameter = 300.0 # units: mm

weird_holder = ContainerRack(
    name="weird arc-shaped tubeholder",
    size_x=diameter,
    size_y=diameter,
    size_z=160.0,
    ordered_items={c.name: c for c in container_holders}
)

In [13]:
facility.assign_child_by_anchor(
    resource=weird_holder,
    parent_anchor=("c","c","b"),
    child_anchor=("c","c", "b")
)

In [14]:
from pylabrobot.resources.corning import cor_falcon_tube_14mL_Rb

In [19]:
import time

In [20]:
for idx, holder in enumerate(weird_holder.get_all_children()):
    print(idx)
    weird_holder[idx] = cor_falcon_tube_14mL_Rb(name=f"tube_{idx}")
    time.sleep(1)


0
1
2
3
4
5
6
7
8
9
10
11


## Ready the STAR(let)

N.B>.: we're using `STARlet` because its smalle size makes this workshop (where we're all on small-screen laptops) easier.
To switch to a STAR in your own lab simply delete the last 3 letters:
`STARlet` -> `STAR`

In [7]:
from pylabrobot.hamilton.star import STARlet

star = STARlet(
    simulation=star_simulation,
    )

In [8]:
facility.assign_child_resource(
    star,
    location=Coordinate(x=800.0, y=0.0, z=0.0)
)

## Run `.setup()` for the Features to be loaded
...from a config file

In [9]:
await star.setup()

2026-09-28 10:05:18,748 - pylabrobot.hamilton.star.driver.master - INFO - [Hamilton STAR] Connected on simulation (no link)
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Arms: 1
  Configuration: 30 slots
  Autoload: 1D barcode scanner
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 800.2 mm, workspace -323.2 to 977.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


## Same for Prep

In [10]:
prep_simulation = True  # True or False

In [11]:
from pylabrobot.hamilton.prep import Prep

prep = Prep(
    simulation=prep_simulation,
    )

facility.assign_child_resource(
    prep,
    location=Coordinate(x=0.0, y=0.0, z=0.0)
)

In [12]:
await prep.setup()
await prep.lights.turn_on()

2026-09-28 10:09:21,828 - pylabrobot.hamilton.prep.driver.master - INFO - tips held at setup: none
2026-09-28 10:09:21,830 - pylabrobot.hamilton.prep.driver.master - WARNING - the 8-channel head is not raised at setup: no move of its Z alone is known
2026-09-28 10:09:21,832 - pylabrobot.hamilton.prep.driver.master - INFO - [Hamilton Prep] Connected on simulation (no link)

  Serial: PRPBC1317

  Firmware: MLPrep Runtime V3.0.20.675 99020-02 Rev N

  Configuration: 
 - enclosure installed,  - safe speeds off,
 - traverse height 167.5 mm
  Deck: x 0.00 to 299.00 mm, y -9.00 to 385.00 mm, z 19.50 to 167.50 mm; 6 sites, 3 waste sites
  Pipettes: 2, v2 aspirate/dispense
    channel 0 (rear): firmware Channel1ml Runtime V1.6.2.270 98228-02 Rev U, x 0.45 to 299.45 mm, y 0.00 to 385.00 mm, z 19.50 to 167.50 mm
    channel 1 (front): firmware Channel1ml Runtime V1.6.2.270 98228-02 Rev U, x 0.45 to 299.45 mm, y -9.00 to 376.00 mm, z 19.50 to 167.50 mm
  8-channel head: installed
  heater shaker:

---
# Configuration JSON
...what the device *is*, as a file

At `setup()`, `discover()` asks the device what is fitted: channels, heads, iSWAP, autoload, firmware.
The answer is kept on `.driver.configuration`, and `save_configuration()` writes it to a JSON file.

That one file is used two ways:
1. **Simulation:** a simulated device answers exactly as the file says, no robot needed.
2. **Real device:** passed as `declared_configuration_json=`, it is checked against what the device answers, and `setup()` stops if they differ.

## What `setup()` read

In [13]:
for device in (star, prep):
    c = device.driver.configuration
    print(device.name, "|", c.serial_number, "|", c.firmware_version)

STARlet | STARlet1 | 7.6S 25 2021_11_05 (GRU C0)
Prep | PRPBC1317 | MLPrep Runtime V3.0.20.675 99020-02 Rev N


In [14]:
c = star.driver.configuration
{"channels": c.num_pip_channels, "96-head": c.head96_installed, "iSWAP": c.kb_iswap_installed, "autoload": c.autoload_installed}

{'channels': 8, '96-head': True, 'iSWAP': True, 'autoload': True}

In [15]:
p = prep.driver.configuration
{"channels": p.num_channels, "8-channel head": p.head8_installed, "deck sites": len(p.deck_sites)}

{'channels': 2, '8-channel head': True, 'deck sites': 6}

## Where a simulation's answers come from

In [16]:
import os

print(os.path.basename(star.driver.declared_configuration_json or "none declared"))
print(os.path.basename(prep.driver.declared_configuration_json or "none declared"))

starlet_legacy_2021_8ch_head96_autoload1D.json
prep_PRPBC1317_v3_0_20.json


## Save it

On a physical device, you are likely interested to
1. perform a discovery of your device's configuration 
2. save it
3. set your simulation mode to always use your configuration json, so simulation is tailored to your device

In [17]:
import json

os.makedirs("_logs", exist_ok=True)
star_json = f"_logs/{run_identifier}_star_configuration.json"
star.driver.save_configuration(star_json)

with open(star_json) as f:
    saved = json.load(f)

{section: len(fields) for section, fields in saved.items()}  # fields per section

{'device': 53, 'arms': 1, 'autoload': 38}

In [18]:
saved["device"]["num_pip_channels"], saved["device"]["head96_installed"]

(8, True)

## Edit the file, get a different device
The same file with a few values changed: a STARlet with 4 channels, no 96-head and no autoload, and without the extension housing.

In [19]:
saved["device"]["num_pip_channels"] = 4

saved["device"]["head96_installed"] = False  # the device, and the arm that carries it
saved["device"]["left_arm"]["head96_installed"] = False
saved["arms"]["left"].pop("head96", None)

saved["device"]["autoload_installed"] = False
saved.pop("autoload", None)

star_4ch_json = f"{run_identifier}_star_4ch_configuration.json"  # next to this notebook
with open(star_4ch_json, "w") as f:
    json.dump(saved, f, indent=2)

star_4ch = STARlet(
    simulation=True,
    declared_configuration_json=star_4ch_json,
    name="star_4ch",
    extension_housing=False,  # a resource, not in the file
)
await star_4ch.setup()

facility.assign_child_resource(
    star_4ch,
    location=Coordinate(x=800.0, y=-1_100.0, z=0.0)
)

{
    "channels": star_4ch.pipettes.num_channels,
    "96-head": star_4ch.head96 is not None,
    "autoload": star_4ch.autoload is not None,
}

2026-09-28 10:11:38,249 - pylabrobot.hamilton.star.driver.master - INFO - [Hamilton STAR] Connected on simulation (no link)
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, iswap 4.1S 2011-12-19
  Arms: 1
  Configuration: 30 slots
  Autoload: none
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 800.2 mm, workspace -323.2 to 977.2 mm
      channels: 4 (1000uL) | 96-head: none | 384-head: none | iSWAP: wide gripper


{'channels': 4, '96-head': False, 'autoload': False}

In [20]:
await star_4ch.stop()

## On the real device
Save the configuration once from your robot, keep the file next to your protocol, and declare it:

```python
star = STARlet(declared_configuration_json="our_star.json")
await star.setup()  # stops if the STAR answers differently
```

`setup()` compares what is fitted: channels, heads, iSWAP, autoload. Serial number and firmware are not compared,
so one file serves every STARlet of the same build.

Declare the file we just edited on our 8-channel STARlet, and setup stops before anything moves:

```
ValueError: the declared configuration does not describe this device:
  num_pip_channels: declared 4, device answers 8
  head96_installed: declared False, device answers True
  autoload_installed: declared False, device answers True
  left_arm.head96_installed: declared False, device answers True
```

---
# Shutdown

In [21]:
await star.stop()

In [22]:
await prep.stop()

2026-09-28 10:12:51,309 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client stopped


In [23]:
await viewer.stop()